# Laboratorio de métodos de búsqueda

|                |   |
:----------------|---|
| **Nombre**     |  Claudio Alejo Encarnación Martínez |
| **Fecha**      |  29 octubre 2026 |
| **Expediente** |  750597 | 

Modelos como regresión logística, máquinas de vectores de soporte, y preceptrones multicapa tienen una serie de argumentos que necesitan valores dados antes de poder usarse para ajustarse a un dataset. Estos argumentos se llaman hiperparámetros.

¿Cuál es la diferencia entre un parámetro y un hiperparámetro?

- **RESPUESTA**

Un parámetro se calcula y se ajusta automáticamente por el algoritmo según los datos. Ejemplo: los coeficientes en una regresión lineal.

Mientras que el híperparámetro se ajusta manual o arbitrariamente antes de entrenar el modelo. Ejemplo: el número de iteraciones a correr.

Este laboratorio se centra en métodos de búsqueda para encontrar los mejores hiperparámetros para un modelo dado. Imagina que estás jugando batalla naval:

<center><img src="https://cdn11.bigcommerce.com/s-nehqfptb/images/stencil/670x993/products/40096/100569/battleship-game__73281.1768238533.jpg?c=2"></center>

Una forma de jugar podría ser ir de cuadro en cuadro hasta probar todo el tablero. El problema aquí es que se juega a contrarreloj, debido a que tu oponente puede hundir tus barcos antes de que pruebes todo el tablero.

Si no se tiene la preocupación/limitante de tiempo, probar todas las combinaciones es una opción:

https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html

Entra a la documentación de `GridSearchCV` de `sklearn` y describe cómo funciona este objeto (entradas, salidas, parámetros).

El objeto recibe un estimador y un diccionario (o lista de diccionarios) que define la "rejilla" de parámetros a probar. Para cada combinación posible, entrena el modelo usando validación cruzada (por defecto, 5 particiones) y calcula una métrica de evaluación. Al finalizar, selecciona la combinación con la puntuación media más alta en los conjuntos de validación.

Entradas (parámetros principales del constructor):

- `estimator`: El modelo base a optimizar (ej. SVC(), RandomForestClassifier()). Debe implementar fit y predict .
- `param_grid`: Diccionario o lista de diccionarios con los nombres de los parámetros como claves y listas de valores a probar. Ej: {'C': [1, 10], 'kernel': ('linear', 'rbf')} .
- `scoring`: Métrica usada para evaluar las combinaciones. Si es None, se usa el método score del estimador .
- `cv`: Esquema de validación cruzada (número de folds o un objeto generador de particiones) .
- `refit`: Booleano (por defecto True). Si es True, reentrena el mejor modelo con todos los datos.
- `n_jobs`: Número de trabajos en paralelo (-1 usa todos los núcleos).
- `verbose`: Nivel de detalle de los mensajes durante el proceso.
- `error_score`: Valor a asignar si falla el entrenamiento de una combinación (por defecto 'raise', que lanza la excepción).

Salidas (atributos principales tras llamar a fit):

- `best_estimator_`: El modelo (ya entrenado con todos los datos) que obtuvo la mejor puntuación. Solo disponible si refit=True .
- `best_score_`: La puntuación media de validación cruzada del mejor modelo.
- `best_params_`: Diccionario con los hiperparámetros que lograron el mejor resultado .
- `best_index_`: Índice (dentro de cv_results_) que corresponde a la mejor combinación .
- `cv_results_`: Diccionario con arrays detallados de todos los resultados (puntuaciones por fold, tiempos, parámetros, rankings, etc.) .
- `classes_`: Etiquetas de clase (solo para clasificadores y si refit=True).
- `n_features_in_`: Número de características vistas durante el fit.

Otra forma de "buscar" podría ser tirar al azar y ver qué pasa. Esto sería otra forma de utilizar la fuerza bruta para encontrar información. 

https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html

Entra a la documentación de `RandomizedSearchCV` de `sklearn` y describe cómo funciona este objeto (entradas, salidas, parámetros). 

- En lugar de probar todas las combinaciones, muestrea al azar un número fijo de combinaciones del espacio de hiperparámetros y evalúa cada una con validación cruzada. Es como "disparar al azar" en la batalla naval un número limitado de veces.

- Entradas principales:
  - `estimator`: el modelo o pipeline.
  - `param_distributions`: diccionario donde cada hiperparámetro tiene una **lista** (se muestrea uniformemente de ella) o una **distribución** de `scipy.stats` (por ejemplo `uniform`, `loguniform`, `randint`), de la que se muestrea con el método `.rvs()`.
  - `n_iter`: número de combinaciones a probar (por defecto 10). Es el parámetro que controla el costo computacional.
  - `scoring`, `cv`, `n_jobs`, `refit`, `verbose`, `return_train_score`: funcionan igual que en `GridSearchCV`.
  - `random_state`: semilla para que el muestreo sea reproducible.

- **Salidas / atributos:** los mismos que `GridSearchCV`: `best_params_`, `best_score_`, `best_estimator_`, `cv_results_`, además de `.predict()`, `.predict_proba()`, etc.

- Ventajas y desventajas:*
  - Ventaja: el costo lo decidimos nosotros (`n_iter` × folds), sin importar qué tan grande sea el espacio. Permite usar distribuciones continuas, y en la práctica suele encontrar resultados muy cercanos al grid con muchas menos evaluaciones.
  - Desventaja: no garantiza encontrar la mejor combinación. Además no aprende de los intentos anteriores, porque cada tiro es independiente.

**Nota:** RandomizedSearchCV prueba puramente al azar. Si encontrara en uno de sus tiros que se atinó a un barco, no cambia el algoritmo de búsqueda para seguir buscando en esa zona. Para un proceso de búsqueda más refinado necesitaríamos un algoritmo que cambie su forma de decidir a partir de información anterior.

Revisa la documentación de Optuna (https://optuna.org/). Instala la librería y escribe una guía de cómo se debe utlizar para un dataset dado.

Optuna es una librería de optimización de hiperparámetros que hace una búsqueda secuencial e inteligente. Por defecto usa el algoritmo TPE (*Tree-structured Parzen Estimator*), que usa los resultados de los intentos anteriores para decidir qué combinación probar después. Sería el jugador de batalla naval que, cuando le atina a un barco, sigue tirando cerca de esa zona.

Conceptos clave:
- Study: el proceso completo de optimización. Guarda todos los intentos.
- Trial: un intento, es decir, una combinación de hiperparámetros evaluada.
- Función objetivo: función que recibe un `trial`, propone hiperparámetros, entrena el modelo y regresa el valor a optimizar (por ejemplo el ROC-AUC promedio de CV).
- Sampler: el algoritmo que decide los siguientes valores (`TPESampler` por defecto, también `RandomSampler`, `GridSampler`, `CmaEsSampler`).

Pasos para usarlo con un dataset:

1. Instalar: `pip install optuna`
2. Cargar y preparar los datos (`X`, `y`) y definir la validación cruzada.
3. Definir la función objetivo y, dentro de ella, el espacio de búsqueda con los métodos `suggest`:
   - `trial.suggest_int('nombre', low, high)` → enteros
   - `trial.suggest_float('nombre', low, high, log=True)` → continuos (con `log=True` para escalas logarítmicas)
   - `trial.suggest_categorical('nombre', [opciones])` → categóricos
   
   Se pueden usar `if` para hiperparámetros condicionales (ej: solo proponer neuronas de la capa 3 si hay 3 capas). Esto es algo que Grid y Random no permiten de forma tan natural.
4. Crear el estudio indicando si se maximiza o minimiza: `study = optuna.create_study(direction='maximize')`
5. Optimizar: `study.optimize(objetivo, n_trials=50)`
6. Revisar resultados: `study.best_params`, `study.best_value`, `study.best_trial` y `study.trials_dataframe()`.
7. Reentrenar el modelo final con `study.best_params` sobre todos los datos de entrenamiento.
8. (Opcional) Visualizar con `optuna.visualization` (historial de optimización, importancia de hiperparámetros, etc.).

### Ejercicio

Utiliza el dataset `College.csv` para intentar predecir si una universidad es privada a partir del resto de los factores.

Utiliza un perceptrón multicapa con hiperparámetros:
- Número de capas intermedias --> {2, 3, 4}
- Número de neuronas en capa 1 --> [5, 10]
- Número de neuronas en capa 2 --> [10, 20]
- Número de neuronas en capa 3 (si existe) --> [5, 20]
- Número de neuronas en capa 4 (si existe) --> [5, 10]

Utiliza k-folds cross-validation con $k=10$; optimiza el ROC-AUC de test.

Compara los 3 diferentes métodos de búsqueda descritos en este laboratorio.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold, cross_val_score
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import optuna

### EDA

In [2]:
df = pd.read_csv('/Users/claudioalejoencarnacionmartinez/Documents/SEMESTRE OTOÑO 2026/LAB APRENDIZAJE ESTADÍSTICO/LABORATORIO-DE-APRENDIZAJE-ESTAD-STICO/College.csv')
df.head()

,Unnamed: 0,Private,Apps,Accept,Enroll,Top10perc,Top25perc,F.Undergrad,P.Undergrad,Outstate,Room.Board,Books,Personal,PhD,Terminal,S.F.Ratio,perc.alumni,Expend,Grad.Rate
0,Abilene Christian University,Yes,1660,1232,721,23,52,2885,537,7440,3300,450,2200,70,78,18.1,12,7041,60
1,Adelphi University,Yes,2186,1924,512,16,29,2683,1227,12280,6450,750,1500,29,30,12.2,16,10527,56
2,Adrian College,Yes,1428,1097,336,22,50,1036,99,11250,3750,400,1165,53,66,12.9,30,8735,54
3,Agnes Scott College,Yes,417,349,137,60,89,510,63,12960,5450,450,875,92,97,7.7,37,19016,59
4,Alaska Pacific University,Yes,193,146,55,16,44,249,869,7560,4120,800,1500,76,72,11.9,2,10922,15


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 777 entries, 0 to 776
Data columns (total 19 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Unnamed: 0   777 non-null    str    
 1   Private      777 non-null    str    
 2   Apps         777 non-null    int64  
 3   Accept       777 non-null    int64  
 4   Enroll       777 non-null    int64  
 5   Top10perc    777 non-null    int64  
 6   Top25perc    777 non-null    int64  
 7   F.Undergrad  777 non-null    int64  
 8   P.Undergrad  777 non-null    int64  
 9   Outstate     777 non-null    int64  
 10  Room.Board   777 non-null    int64  
 11  Books        777 non-null    int64  
 12  Personal     777 non-null    int64  
 13  PhD          777 non-null    int64  
 14  Terminal     777 non-null    int64  
 15  S.F.Ratio    777 non-null    float64
 16  perc.alumni  777 non-null    int64  
 17  Expend       777 non-null    int64  
 18  Grad.Rate    777 non-null    int64  
dtypes: float64(1), int6

In [4]:
df.describe()

,Apps,Accept,Enroll,Top10perc,Top25perc,F.Undergrad,P.Undergrad,Outstate,Room.Board,Books,Personal,PhD,Terminal,S.F.Ratio,perc.alumni,Expend,Grad.Rate
count,777.000000,777.000000,777.000000,777.000000,777.000000,777.000000,777.000000,777.000000,777.000000,777.000000,777.000000,777.000000,777.000000,777.000000,777.000000,777.000000,777.00000
mean,3001.638353,2018.804376,779.972973,27.558559,55.796654,3699.907336,855.298584,10440.669241,4357.526384,549.380952,1340.642214,72.660232,79.702703,14.089704,22.743887,9660.171171,65.46332
std,3870.201484,2451.113971,929.176190,17.640364,19.804778,4850.420531,1522.431887,4023.016484,1096.696416,165.105360,677.071454,16.328155,14.722359,3.958349,12.391801,5221.768440,17.17771
min,81.000000,72.000000,35.000000,1.000000,9.000000,139.000000,1.000000,2340.000000,1780.000000,96.000000,250.000000,8.000000,24.000000,2.500000,0.000000,3186.000000,10.00000
25%,776.000000,604.000000,242.000000,15.000000,41.000000,992.000000,95.000000,7320.000000,3597.000000,470.000000,850.000000,62.000000,71.000000,11.500000,13.000000,6751.000000,53.00000
50%,1558.000000,1110.000000,434.000000,23.000000,54.000000,1707.000000,353.000000,9990.000000,4200.000000,500.000000,1200.000000,75.000000,82.000000,13.600000,21.000000,8377.000000,65.00000
75%,3624.000000,2424.000000,902.000000,35.000000,69.000000,4005.000000,967.000000,12925.000000,5050.000000,600.000000,1700.000000,85.000000,92.000000,16.500000,31.000000,10830.000000,78.00000
max,48094.000000,26330.000000,6392.000000,96.000000,100.000000,31643.000000,21836.000000,21700.000000,8124.000000,2340.000000,6800.000000,103.000000,100.000000,39.800000,64.000000,56233.000000,118.00000


In [5]:
# la primera columna trae el nombre de la universidad, no sirve como variable
df = df.drop(columns=['Unnamed: 0'])

print(df.shape)
print(df['Private'].value_counts())
df.isnull().sum()

(777, 18)
Private
Yes    565
No     212
Name: count, dtype: int64


Private        0
Apps           0
Accept         0
Enroll         0
Top10perc      0
Top25perc      0
F.Undergrad    0
P.Undergrad    0
Outstate       0
Room.Board     0
Books          0
Personal       0
PhD            0
Terminal       0
S.F.Ratio      0
perc.alumni    0
Expend         0
Grad.Rate      0
dtype: int64

### Definición de variables

In [6]:
X = df.drop(columns=['Private'])
y = (df['Private'] == 'Yes').astype(int)

La clase está desbalanceada, por eso se usa `StratifiedKFold` para que cada fold conserve la misma proporción. El ROC-AUC también es buena métrica en este caso porque no depende de un umbral de clasificación.

### Pipeline del modelo

In [7]:
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=67)

modelo = Pipeline([('scaler', StandardScaler()),
                ('mlp', MLPClassifier(max_iter=1000, random_state=67))])

### Construcción de las capas a explorar

In [8]:
capas1 = list(range(5, 11))    # 5 a 10
capas2 = list(range(10, 21))   # 10 a 20
capas3 = list(range(5, 21))    # 5 a 20
capas4 = list(range(5, 11))    # 5 a 10

arquitecturas = []
for a in capas1:
    for b in capas2:
        arquitecturas.append((a, b))                  # 2 capas
        for c in capas3:
            arquitecturas.append((a, b, c))           # 3 capas
            for d in capas4:
                arquitecturas.append((a, b, c, d))    # 4 capas

print('Total de arquitecturas:', len(arquitecturas))
print('2 capas:', sum(len(t) == 2 for t in arquitecturas))
print('3 capas:', sum(len(t) == 3 for t in arquitecturas))
print('4 capas:', sum(len(t) == 4 for t in arquitecturas))

Total de arquitecturas: 7458
2 capas: 66
3 capas: 1056
4 capas: 6336


## Optuna

### Definición de la función objetivo con proposición de híperparámetros

In [9]:
def objetivo(trial):
    n_capas = trial.suggest_int('n_capas', 2, 4)
    n1 = trial.suggest_int('n1', 5, 10)
    n2 = trial.suggest_int('n2', 10, 20)
    capas = [n1, n2]
    if n_capas >= 3:
        capas.append(trial.suggest_int('n3', 5, 20))
    if n_capas == 4:
        capas.append(trial.suggest_int('n4', 5, 10))

    modelo_opt = Pipeline([
        ('scaler', StandardScaler()),
        ('mlp', MLPClassifier(hidden_layer_sizes=tuple(capas), max_iter=1000, random_state=42))
    ])
    scores = cross_val_score(modelo_opt, X, y, cv=kf, scoring='roc_auc', n_jobs=-1)
    return scores.mean()

### Estudio

In [10]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

estudio = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))

inicio = time.time()
estudio.optimize(objetivo, n_trials=50)
tiempo_optuna = time.time() - inicio

mejor = estudio.best_params
arq_optuna = [mejor['n1'], mejor['n2']]
if mejor['n_capas'] >= 3:
    arq_optuna.append(mejor['n3'])
if mejor['n_capas'] == 4:
    arq_optuna.append(mejor['n4'])

print('Mejores parámetros:', mejor)
print('Mejor arquitectura:', tuple(arq_optuna))
print('Mejor ROC-AUC CV:', round(estudio.best_value, 4))
print('Tiempo (s):', round(tiempo_optuna, 1))

Mejores parámetros: {'n_capas': 2, 'n1': 6, 'n2': 19}
Mejor arquitectura: (6, 19)
Mejor ROC-AUC CV: 0.975
Tiempo (s): 1128.4


In [11]:
res_optuna = estudio.trials_dataframe()
print(res_optuna['params_n_capas'].value_counts())
res_optuna[['number', 'value', 'params_n_capas', 'params_n1', 'params_n2', 'params_n3', 'params_n4']].sort_values('value', ascending=False).head(10)

params_n_capas
2    32
3    14
4     4
Name: count, dtype: int64


,number,value,params_n_capas,params_n1,params_n2,params_n3,params_n4
16,16,0.974952,2,6,19,NaN,NaN
15,15,0.974952,2,6,19,NaN,NaN
41,41,0.974952,2,6,19,NaN,NaN
26,26,0.974952,2,6,19,NaN,NaN
33,33,0.974952,2,6,19,NaN,NaN
42,42,0.974952,2,6,19,NaN,NaN
31,31,0.974952,2,6,19,NaN,NaN
46,46,0.973529,2,6,16,NaN,NaN
17,17,0.970489,2,6,20,NaN,NaN
24,24,0.970489,2,6,20,NaN,NaN


## RandomSearchCV

In [13]:
param_dist = {'mlp__hidden_layer_sizes': arquitecturas}

random_search = RandomizedSearchCV(modelo, param_dist, n_iter=50, scoring='roc_auc',
                                   cv=kf, n_jobs=-1, random_state=42, verbose=1)

inicio = time.time()
random_search.fit(X, y)
tiempo_random = time.time() - inicio

print('Mejor arquitectura:', random_search.best_params_)
print('Mejor ROC-AUC CV:', round(random_search.best_score_, 4))
print('Tiempo (s):', round(tiempo_random, 1))

Fitting 10 folds for each of 50 candidates, totalling 500 fits


Mejor arquitectura: {'mlp__hidden_layer_sizes': (10, 10, 12)}
Mejor ROC-AUC CV: 0.9663
Tiempo (s): 1056.7


In [15]:
res_random = pd.DataFrame(random_search.cv_results_)
res_random['n_capas'] = res_random['param_mlp__hidden_layer_sizes'].apply(len)
print(res_random['n_capas'].value_counts())
res_random[['param_mlp__hidden_layer_sizes', 'mean_test_score', 'std_test_score', 'rank_test_score']].sort_values('rank_test_score').head(10)

n_capas
4    42
3     7
2     1
Name: count, dtype: int64


,param_mlp__hidden_layer_sizes,mean_test_score,std_test_score,rank_test_score
8,"(10, 10, 12)",0.966285,0.027095,1
22,"(10, 16, 12, 10)",0.963962,0.026898,2
12,"(9, 15, 10, 9)",0.963864,0.026366,3
43,"(7, 12, 8)",0.963278,0.033952,4
48,"(10, 16, 16, 8)",0.962801,0.023600,5
34,"(5, 14, 8)",0.962257,0.026788,6
26,"(10, 11, 18)",0.961375,0.029846,7
39,"(5, 18, 15)",0.961364,0.028154,8
17,"(5, 11, 7, 6)",0.959333,0.028065,9
13,"(10, 10, 7, 5)",0.958735,0.027375,10


In [16]:
comparacion = pd.DataFrame({
    'Método': ['RandomizedSearchCV', 'Optuna (TPE)'],
    'Mejor arquitectura': [random_search.best_params_['mlp__hidden_layer_sizes'],
                           tuple(arq_optuna)],
    'Mejor ROC-AUC CV': [random_search.best_score_, estudio.best_value],
    'Combinaciones evaluadas': [len(res_random), len(res_optuna)],
    'Modelos entrenados': [len(res_random)*10, len(res_optuna)*10],
    'Tiempo (s)': [tiempo_random, tiempo_optuna]
})
comparacion['Mejor ROC-AUC CV'] = comparacion['Mejor ROC-AUC CV'].round(4)
comparacion['Tiempo (s)'] = comparacion['Tiempo (s)'].round(1)
comparacion

,Método,Mejor arquitectura,Mejor ROC-AUC CV,Combinaciones evaluadas,Modelos entrenados,Tiempo (s)
0,RandomizedSearchCV,"(10, 10, 12)",0.9663,50,500,1056.7
1,Optuna (TPE),"(6, 19)",0.9750,50,500,1128.4
